# Multi-Agent · Day 34 — **AutoGen & Semantic Kernel**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~2 hrs

LangGraph (Days 31–33) makes you draw the graph: *you* decide the edges. **AutoGen** takes the opposite bet — you give agents personas and a shared conversation, and the **conversation itself is the control flow**. Two agents talk turn-by-turn until a termination condition fires. The canonical pair is `AssistantAgent` (writes) ↔ `UserProxyAgent` (executes + feeds results back).

We can't ship an OpenAI key in a notebook, so below is a **faithful, runnable mock** of AutoGen's core loop — same roles, same turn-taking, same termination logic — plus the one comparison an AVP is actually asked to make: *AutoGen vs LangGraph, when?*

Today:
1. The **two-agent conversation loop** (assistant ↔ executor) that defines AutoGen.
2. **Code generation + execution** — the pattern's killer app, with a real sandbox.
3. **Termination** — how a free-form chat knows it's done.
4. **AutoGen vs LangGraph** decision table + where Semantic Kernel fits.

## 1. The two-agent conversation loop

AutoGen's runtime is a message list two agents append to in turns. `AssistantAgent.generate_reply()` produces text; `UserProxyAgent.generate_reply()` either runs code found in that text or relays a human/tool result. Strip the LLM out and the *shape* is this.

In [1]:
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class Message:
    role: str          # agent name that produced it
    content: str

@dataclass
class ConversableAgent:
    """Minimal stand-in for autogen.ConversableAgent."""
    name: str
    reply_fn: Callable[[list[Message]], Message]   # in real AutoGen this is the LLM

    def generate_reply(self, history: list[Message]) -> Message:
        return self.reply_fn(history)

def initiate_chat(a: ConversableAgent, b: ConversableAgent,
                  first: str, max_turns: int = 6) -> list[Message]:
    """Turn-taking loop: a speaks, b replies, a replies... (autogen's core)."""
    history = [Message(role="user", content=first)]
    speakers = [a, b]
    for turn in range(max_turns):
        speaker = speakers[turn % 2]
        msg = speaker.generate_reply(history)
        history.append(msg)
        print(f"[{speaker.name}] {msg.content.splitlines()[0][:70]}")
        if msg.content.strip().endswith("TERMINATE"):
            break
    return history

☝ Two things define AutoGen and both are here: **(a)** there is no graph — `initiate_chat` just alternates speakers, so the "flow" is emergent from what each agent says; **(b)** a plain `TERMINATE` sentinel in the message text ends the run. That flexibility is the whole pitch and also the whole risk — nothing structurally stops the two from chatting forever except `max_turns`.

## 2. Code generation + execution — the pattern that made AutoGen famous

The `AssistantAgent` writes Python in a ```` ```python ```` block; the `UserProxyAgent` **extracts and runs it**, then feeds stdout back. That execution feedback loop is why AutoGen is strong at "compute this, check it, fix it" tasks. Banking task: size a loan repayment.

In [2]:
import re

# --- The assistant: proposes code, then verifies the returned result. ---
def assistant_reply(history: list[Message]) -> Message:
    last = history[-1].content
    if "RESULT:" in last:                       # executor gave us an answer -> verify + stop
        val = float(re.search(r"RESULT:\s*([\d.]+)", last).group(1))
        ok = "within policy" if val <= 2000 else "EXCEEDS affordability cap"
        return Message("assistant", f"Monthly repayment GBP {val:.2f} — {ok}. TERMINATE")
    # first turn: emit code for the executor
    return Message("assistant", "Compute the monthly repayment.\n"
        "```python\n"
        "P, annual_rate, years = 40_000, 0.079, 3\n"
        "r = annual_rate/12; n = years*12\n"
        "m = P*r*(1+r)**n/((1+r)**n - 1)\n"
        "print(f'RESULT: {m:.2f}')\n"
        "```")

# --- The user proxy: finds a code block, executes it in a scratch namespace. ---
def make_executor():
    def executor_reply(history: list[Message]) -> Message:
        code_block = re.search(r"```python\n(.*?)```", history[-1].content, re.S)
        if not code_block:
            return Message("executor", "No code found. TERMINATE")
        import io, contextlib
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            exec(code_block.group(1), {})       # sandbox namespace = fresh dict
        return Message("executor", buf.getvalue().strip())
    return executor_reply

assistant = ConversableAgent("assistant", assistant_reply)
user_proxy = ConversableAgent("user_proxy", make_executor())

hist = initiate_chat(assistant, user_proxy, "Can this customer afford a GBP 40k loan?")
print("\nFINAL:", hist[-1].content)

[assistant] Compute the monthly repayment.
[user_proxy] RESULT: 1251.61
[assistant] Monthly repayment GBP 1251.61 — within policy. TERMINATE

FINAL: Monthly repayment GBP 1251.61 — within policy. TERMINATE


☝ The loop ran: assistant → code, executor → `RESULT: 1250.11`, assistant → verify + `TERMINATE`. Real AutoGen swaps `assistant_reply` for an LLM call and runs the executor inside Docker (`code_execution_config={"use_docker": True}`) — **never** bare `exec` on model-written code in production, since the model can be prompt-injected into writing anything. That Docker default is the security answer to "you let an LLM write code you then run".

## 3. Termination — how a free-form chat stops

With no graph, termination is a first-class concern. AutoGen offers three levers; you almost always combine them so one always fires.

In [3]:
def is_termination(msg: Message) -> bool:
    return msg.content.strip().endswith("TERMINATE")

TERMINATION_LEVERS = {
    "sentinel string": "is_termination_msg=lambda m: 'TERMINATE' in m['content']",
    "max turns":       "max_turns=N on initiate_chat / max_consecutive_auto_reply",
    "human in loop":   "human_input_mode='TERMINATE' — ask a person before stopping",
}
for lever, api in TERMINATION_LEVERS.items():
    print(f"{lever:16} -> {api}")

# Why you combine them: a model that never emits the sentinel must still be capped.
print("\nsentinel fired:", is_termination(hist[-1]))
print("would-cap-at   :", "max_turns=6 (backstop even if sentinel never appears)")

sentinel string  -> is_termination_msg=lambda m: 'TERMINATE' in m['content']
max turns        -> max_turns=N on initiate_chat / max_consecutive_auto_reply
human in loop    -> human_input_mode='TERMINATE' — ask a person before stopping

sentinel fired: True
would-cap-at   : max_turns=6 (backstop even if sentinel never appears)


☝ For a regulated deployment the sentinel alone is unacceptable — a confused or injected model may never say `TERMINATE`, so `max_turns` is the non-negotiable backstop, exactly like the `recursion_limit` you put on the LangGraph supervisor (Day 33). `human_input_mode` is how AutoGen inserts the **human approval gate**; you'll build the LangGraph equivalent in the Day 38 capstone.

## 4. AutoGen vs LangGraph — the decision you'll defend

Both orchestrate multiple agents; they optimise for different things. Pick per-workload, not per-religion.

In [4]:
comparison = [
    # (dimension,           AutoGen,                          LangGraph)
    ("Control flow",        "emergent from conversation",     "explicit graph you draw"),
    ("Best at",             "code-gen, exploratory, research", "auditable production workflows"),
    ("Determinism",         "low — chat can wander",          "high — edges are fixed"),
    ("State",               "the message list",               "typed State + reducers"),
    ("Audit story",         "read the transcript",            "graph + checkpoints + trace"),
    ("Human gate",          "human_input_mode",               "interrupt_before + checkpointer"),
    ("Resume after crash",  "manual",                         "thread_id + Saver (built in)"),
]
w = max(len(d) for d, _, _ in comparison)
print(f"{'':{w}}  {'AutoGen':34} LangGraph")
for d, a, l in comparison:
    print(f"{d:{w}}  {a:34} {l}")

                    AutoGen                            LangGraph
Control flow        emergent from conversation         explicit graph you draw
Best at             code-gen, exploratory, research    auditable production workflows
Determinism         low — chat can wander              high — edges are fixed
State               the message list                   typed State + reducers
Audit story         read the transcript                graph + checkpoints + trace
Human gate          human_input_mode                   interrupt_before + checkpointer
Resume after crash  manual                             thread_id + Saver (built in)


☝ The AVP-level answer: **AutoGen for open-ended, code-in-the-loop, and research tasks where you want agents to figure out the path; LangGraph when a regulator will ask to see the path.** Barclays production flows (loan decisioning, fraud triage) are the latter — you want fixed edges, typed state, checkpoints. AutoGen earns its place in the *build* phase (prototyping, letting agents explore a data problem) and as a sub-tool a LangGraph node can call.

**Semantic Kernel** is the Microsoft/.NET-native option: plugins (typed tool functions) + planners, tuned for Azure/enterprise shops. If a team is C#-first on Azure, SK is the path of least resistance; the concepts (tools, planners ≈ supervisor) map straight onto what you've built.

## 5. Recap

| Concept | AutoGen mechanism | Cell |
|---|---|---|
| Flow = conversation | alternating `generate_reply` turns | 1 |
| Code + execution loop | assistant writes ```python```, proxy runs it | 2 |
| Safe execution | Docker sandbox (never bare `exec` in prod) | 2 |
| Termination | sentinel **+** max_turns **+** human gate | 3 |
| AutoGen vs LangGraph | emergent chat vs explicit auditable graph | 4 |
| Semantic Kernel | Azure/.NET plugins + planners | 4 |

**Rule of thumb:** *AutoGen to explore, LangGraph to ship.* Tomorrow (Day 35): Google ADK and the **A2A protocol** — the emerging standard for agents built on *different* frameworks to talk to each other.